# Faster R-CNN vs. DETR on Pascal VOC 2012

This Colab-first notebook provides a corrected, reproducible comparison of a two-stage convolutional detector and an end-to-end transformer detector.

**Comparison contract**

- Official Pascal VOC 2012 train and validation splits
- The same 20 object classes and validation images
- Zero-based class IDs in both pipelines
- The same TorchMetrics `mAP@0.50` evaluation for both models
- Persistent data, checkpoints, metrics, and figures in Google Drive

Start with `QUICK_RUN = True` to verify the pipeline. Use `QUICK_RUN = False` for results that can be published.

## 1. Connect the notebook to the GitHub project

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/zhuoshen1111/object-detection-faster-rcnn-detr.git'
REPO_DIR = Path('/content/object-detection-faster-rcnn-detr')

if 'COLAB_GPU' in os.environ and not (REPO_DIR / 'src' / 'voc.py').exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)

if (REPO_DIR / 'src' / 'voc.py').exists():
    os.chdir(REPO_DIR)
    print('Working directory:', Path.cwd())
else:
    print('Repository files were not found. Clone the repository or run from its root directory.')
    print('Expected:', REPO_DIR)

## 2. Install dependencies

Detectron2 is built against the Torch/CUDA versions in the active Colab runtime, so this cell may take several minutes.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

print('Installation finished. Restart the runtime only if Colab explicitly asks you to.')

## 3. Mount Drive and set the experiment configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/object-detection-faster-rcnn-detr')
DATA_DIR = DRIVE_PROJECT_DIR / 'data'
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'

for directory in (DATA_DIR, OUTPUT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Pipeline check versus publishable experiment.
QUICK_RUN = True
SEED = 42
FORCE_RETRAIN_FRCNN = False
FORCE_RETRAIN_DETR = False

TRAIN_LIMIT = 256 if QUICK_RUN else None
EVAL_LIMIT = 128 if QUICK_RUN else None
FRCNN_MAX_ITER = 100 if QUICK_RUN else 2000
DETR_EPOCHS = 1 if QUICK_RUN else 10
BATCH_SIZE = 2
NUM_WORKERS = 2

RUN_NAME = 'quick' if QUICK_RUN else 'full'
print({
    'run': RUN_NAME,
    'train_limit': TRAIN_LIMIT,
    'eval_limit': EVAL_LIMIT,
    'frcnn_iterations': FRCNN_MAX_ITER,
    'detr_epochs': DETR_EPOCHS,
})

## 4. Imports and reproducibility

In [ ]:
import json
import random
import time
from copy import deepcopy

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from src.voc import (
    VOC_CLASSES,
    build_voc_records,
    to_coco_annotations,
    validate_voc_layout,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 5. Download and verify Pascal VOC 2012

`torchvision` downloads the official train/validation archive and keeps it in Google Drive for later sessions.

In [ ]:
from torchvision.datasets import VOCDetection

VOC_ROOT = DATA_DIR / 'VOCdevkit' / 'VOC2012'
if not VOC_ROOT.exists():
    VOCDetection(root=str(DATA_DIR), year='2012', image_set='trainval', download=True)

validate_voc_layout(VOC_ROOT)

train_records = build_voc_records(
    VOC_ROOT,
    'train',
    include_difficult=False,
    max_samples=TRAIN_LIMIT,
)
val_records = build_voc_records(
    VOC_ROOT,
    'val',
    include_difficult=False,
    max_samples=EVAL_LIMIT,
)

train_ids = {record['voc_id'] for record in train_records}
val_ids = {record['voc_id'] for record in val_records}
assert train_ids.isdisjoint(val_ids), 'Train and validation splits overlap.'
assert train_records and val_records, 'Dataset records are empty.'

print('VOC root:', VOC_ROOT)
print('Training images used:', len(train_records))
print('Validation images used:', len(val_records))
print('Classes:', len(VOC_CLASSES))

### Inspect one ground-truth sample

In [ ]:
def draw_boxes(image, boxes, labels, scores=None, color='lime', threshold=0.0):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    for index, (box, label) in enumerate(zip(boxes, labels)):
        score = None if scores is None else float(scores[index])
        if score is not None and score < threshold:
            continue
        xmin, ymin, xmax, ymax = [float(value) for value in box]
        draw.rectangle((xmin, ymin, xmax, ymax), outline=color, width=3)
        caption = VOC_CLASSES[int(label)]
        if score is not None:
            caption += f' {score:.2f}'
        draw.text((xmin + 3, max(0, ymin - 13)), caption, fill=color)
    return canvas


sample_record = random.choice(train_records)
sample_image = Image.open(sample_record['file_name']).convert('RGB')
sample_figure = draw_boxes(sample_image, sample_record['boxes'], sample_record['labels'])

plt.figure(figsize=(12, 8))
plt.imshow(sample_figure)
plt.title(f"Ground truth: {sample_record['voc_id']}")
plt.axis('off')
plt.show()

## 6. Faster R-CNN with Detectron2

In [ ]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.structures import BoxMode


def as_detectron2_records(records):
    converted = []
    for record in records:
        item = {
            'file_name': record['file_name'],
            'image_id': record['voc_id'],
            'height': record['height'],
            'width': record['width'],
            'annotations': [
                {
                    'bbox': box,
                    'bbox_mode': BoxMode.XYXY_ABS,
                    'category_id': label,
                    'iscrowd': 0,
                }
                for box, label in zip(record['boxes'], record['labels'])
            ],
        }
        converted.append(item)
    return converted


TRAIN_DATASET_NAME = f'voc2012_train_{RUN_NAME}'
VAL_DATASET_NAME = f'voc2012_val_{RUN_NAME}'
d2_train_records = as_detectron2_records(train_records)
d2_val_records = as_detectron2_records(val_records)

for name in (TRAIN_DATASET_NAME, VAL_DATASET_NAME):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

DatasetCatalog.register(TRAIN_DATASET_NAME, lambda records=d2_train_records: deepcopy(records))
DatasetCatalog.register(VAL_DATASET_NAME, lambda records=d2_val_records: deepcopy(records))
for name in (TRAIN_DATASET_NAME, VAL_DATASET_NAME):
    MetadataCatalog.get(name).set(thing_classes=list(VOC_CLASSES))

print('Registered:', TRAIN_DATASET_NAME, VAL_DATASET_NAME)

In [ ]:
def build_faster_rcnn_config():
    cfg = get_cfg()
    config_name = 'COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml'
    cfg.merge_from_file(model_zoo.get_config_file(config_name))
    cfg.DATASETS.TRAIN = (TRAIN_DATASET_NAME,)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_name)
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(VOC_CLASSES)
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.0
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = 0.0025
    cfg.SOLVER.MAX_ITER = FRCNN_MAX_ITER
    cfg.SOLVER.STEPS = []
    cfg.SOLVER.CHECKPOINT_PERIOD = max(50, FRCNN_MAX_ITER // 4)
    cfg.OUTPUT_DIR = str(OUTPUT_DIR / f'faster_rcnn_{RUN_NAME}')
    Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    return cfg


cfg_frcnn = build_faster_rcnn_config()
frcnn_checkpoint = Path(cfg_frcnn.OUTPUT_DIR) / 'model_final.pth'

if FORCE_RETRAIN_FRCNN or not frcnn_checkpoint.exists():
    trainer = DefaultTrainer(cfg_frcnn)
    trainer.resume_or_load(resume=False)
    trainer.train()
else:
    print('Using existing Faster R-CNN checkpoint:', frcnn_checkpoint)

if not frcnn_checkpoint.exists():
    raise FileNotFoundError(f'Faster R-CNN checkpoint was not created: {frcnn_checkpoint}')

cfg_frcnn.MODEL.WEIGHTS = str(frcnn_checkpoint)
frcnn_predictor = DefaultPredictor(cfg_frcnn)

## 7. One evaluation implementation for both models

+TorchMetrics receives absolute `xyxy` boxes for predictions and ground truth. Values returned by TorchMetrics are proportions in `[0, 1]`; the summary converts them to percentages.

In [ ]:
from torchmetrics.detection import MeanAveragePrecision


def make_map50_metric():
    return MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        iou_thresholds=[0.5],
        max_detection_thresholds=[1, 10, 100],
    )


def target_from_record(record):
    boxes = torch.tensor(record['boxes'], dtype=torch.float32).reshape(-1, 4)
    labels = torch.tensor(record['labels'], dtype=torch.int64)
    return {'boxes': boxes, 'labels': labels}


def summarize_metric(metric_output):
    keys = ('map', 'map_50', 'mar_100')
    summary = {}
    for key in keys:
        value = metric_output.get(key)
        if value is not None and value.numel() == 1:
            summary[f'{key}_percent'] = float(value.item() * 100.0)
    return summary


@torch.inference_mode()
def evaluate_faster_rcnn(predictor, records):
    metric = make_map50_metric()
    for record in tqdm(records, desc='Evaluate Faster R-CNN'):
        image_bgr = cv2.imread(record['file_name'])
        if image_bgr is None:
            raise FileNotFoundError(record['file_name'])
        instances = predictor(image_bgr)['instances'].to('cpu')
        prediction = {
            'boxes': instances.pred_boxes.tensor.float(),
            'scores': instances.scores.float(),
            'labels': instances.pred_classes.long(),
        }
        metric.update([prediction], [target_from_record(record)])
    return metric.compute()


frcnn_metric_output = evaluate_faster_rcnn(frcnn_predictor, val_records)
frcnn_metrics = summarize_metric(frcnn_metric_output)
print('Faster R-CNN:', frcnn_metrics)

### Faster R-CNN qualitative prediction

In [ ]:
frcnn_record = random.choice(val_records)
frcnn_image_bgr = cv2.imread(frcnn_record['file_name'])
frcnn_instances = frcnn_predictor(frcnn_image_bgr)['instances'].to('cpu')
frcnn_image_rgb = Image.open(frcnn_record['file_name']).convert('RGB')
frcnn_visual = draw_boxes(
    frcnn_image_rgb,
    frcnn_instances.pred_boxes.tensor.tolist(),
    frcnn_instances.pred_classes.tolist(),
    frcnn_instances.scores.tolist(),
    color='red',
    threshold=0.5,
)

plt.figure(figsize=(12, 8))
plt.imshow(frcnn_visual)
plt.title(f"Faster R-CNN prediction: {frcnn_record['voc_id']}")
plt.axis('off')
plt.show()

## 8. DETR with Hugging Face Transformers

+The image processor receives COCO-style absolute `xywh` annotations. It performs resizing and converts target boxes to the normalized representation expected by DETR.

In [ ]:
from transformers import AutoImageProcessor, DetrForObjectDetection


DETR_MODEL_NAME = 'facebook/detr-resnet-50'
detr_processor = AutoImageProcessor.from_pretrained(DETR_MODEL_NAME)


class VOCDatasetForDETR(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record['file_name']).convert('RGB')
        encoding = self.processor(
            images=image,
            annotations=to_coco_annotations(record),
            return_tensors='pt',
        )
        return {
            'pixel_values': encoding['pixel_values'].squeeze(0),
            'labels': encoding['labels'][0],
            'metric_target': target_from_record(record),
            'record': record,
        }


def detr_collate_fn(batch):
    padded = detr_processor.pad(
        [item['pixel_values'] for item in batch],
        return_tensors='pt',
    )
    return {
        'pixel_values': padded['pixel_values'],
        'pixel_mask': padded.get('pixel_mask'),
        'labels': [item['labels'] for item in batch],
        'metric_targets': [item['metric_target'] for item in batch],
        'records': [item['record'] for item in batch],
    }


train_detr_dataset = VOCDatasetForDETR(train_records, detr_processor)
val_detr_dataset = VOCDatasetForDETR(val_records, detr_processor)

loader_generator = torch.Generator().manual_seed(SEED)
train_detr_loader = DataLoader(
    train_detr_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=detr_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator,
)
val_detr_loader = DataLoader(
    val_detr_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=detr_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

first_batch = next(iter(train_detr_loader))
print('pixel_values:', tuple(first_batch['pixel_values'].shape))
print('pixel_mask:', None if first_batch['pixel_mask'] is None else tuple(first_batch['pixel_mask'].shape))
print('labels in first image:', first_batch['labels'][0].keys())

In [ ]:
id2label = {index: name for index, name in enumerate(VOC_CLASSES)}
label2id = {name: index for index, name in id2label.items()}
detr_checkpoint_dir = OUTPUT_DIR / f'detr_{RUN_NAME}'

if detr_checkpoint_dir.exists() and not FORCE_RETRAIN_DETR:
    detr_model = DetrForObjectDetection.from_pretrained(detr_checkpoint_dir).to(device)
    print('Using existing DETR checkpoint:', detr_checkpoint_dir)
else:
    detr_model = DetrForObjectDetection.from_pretrained(
        DETR_MODEL_NAME,
        num_labels=len(VOC_CLASSES),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)

    backbone_parameters = []
    other_parameters = []
    for name, parameter in detr_model.named_parameters():
        if not parameter.requires_grad:
            continue
        if 'backbone' in name:
            backbone_parameters.append(parameter)
        else:
            other_parameters.append(parameter)

    optimizer = torch.optim.AdamW(
        [
            {'params': other_parameters, 'lr': 1e-4},
            {'params': backbone_parameters, 'lr': 1e-5},
        ],
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=max(1, DETR_EPOCHS // 2),
        gamma=0.1,
    )

    history = []
    for epoch in range(DETR_EPOCHS):
        detr_model.train()
        running_loss = 0.0
        for batch in tqdm(train_detr_loader, desc=f'DETR epoch {epoch + 1}/{DETR_EPOCHS}'):
            pixel_values = batch['pixel_values'].to(device)
            pixel_mask = batch['pixel_mask']
            if pixel_mask is not None:
                pixel_mask = pixel_mask.to(device)
            labels = [
                {key: value.to(device) if isinstance(value, torch.Tensor) else value for key, value in target.items()}
                for target in batch['labels']
            ]

            optimizer.zero_grad(set_to_none=True)
            outputs = detr_model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels,
            )
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(detr_model.parameters(), max_norm=0.1)
            optimizer.step()
            running_loss += float(outputs.loss.item())

        scheduler.step()
        epoch_loss = running_loss / max(1, len(train_detr_loader))
        history.append({'epoch': epoch + 1, 'train_loss': epoch_loss})
        print(f'Epoch {epoch + 1}: train_loss={epoch_loss:.4f}')

    detr_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    detr_model.save_pretrained(detr_checkpoint_dir)
    detr_processor.save_pretrained(detr_checkpoint_dir)
    (detr_checkpoint_dir / 'training_history.json').write_text(json.dumps(history, indent=2))

## 9. Evaluate DETR on the same validation records

In [ ]:
@torch.inference_mode()
def evaluate_detr(model, loader):
    metric = make_map50_metric()
    model.eval()
    for batch in tqdm(loader, desc='Evaluate DETR'):
        pixel_values = batch['pixel_values'].to(device)
        pixel_mask = batch['pixel_mask']
        if pixel_mask is not None:
            pixel_mask = pixel_mask.to(device)

        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
        target_sizes = torch.tensor(
            [[record['height'], record['width']] for record in batch['records']],
            dtype=torch.int64,
            device=device,
        )
        processed = detr_processor.post_process_object_detection(
            outputs,
            threshold=0.0,
            target_sizes=target_sizes,
        )
        predictions = [
            {
                'boxes': result['boxes'].detach().cpu().float(),
                'scores': result['scores'].detach().cpu().float(),
                'labels': result['labels'].detach().cpu().long(),
            }
            for result in processed
        ]
        metric.update(predictions, batch['metric_targets'])
    return metric.compute()


detr_metric_output = evaluate_detr(detr_model, val_detr_loader)
detr_metrics = summarize_metric(detr_metric_output)
print('DETR:', detr_metrics)

### DETR qualitative prediction

In [ ]:
@torch.inference_mode()
def predict_detr(record, threshold=0.5):
    image = Image.open(record['file_name']).convert('RGB')
    inputs = detr_processor(images=image, return_tensors='pt').to(device)
    outputs = detr_model(**inputs)
    target_sizes = torch.tensor([[record['height'], record['width']]], device=device)
    result = detr_processor.post_process_object_detection(
        outputs,
        threshold=threshold,
        target_sizes=target_sizes,
    )[0]
    return image, {key: value.detach().cpu() for key, value in result.items()}


detr_record = random.choice(val_records)
detr_image, detr_prediction = predict_detr(detr_record, threshold=0.5)
detr_visual = draw_boxes(
    detr_image,
    detr_prediction['boxes'].tolist(),
    detr_prediction['labels'].tolist(),
    detr_prediction['scores'].tolist(),
    color='yellow',
    threshold=0.5,
)

plt.figure(figsize=(12, 8))
plt.imshow(detr_visual)
plt.title(f"DETR prediction: {detr_record['voc_id']}")
plt.axis('off')
plt.show()

## 10. Save the standardized comparison

In [ ]:
comparison = {
    'run_name': RUN_NAME,
    'quick_run': QUICK_RUN,
    'seed': SEED,
    'dataset': 'Pascal VOC 2012',
    'train_split': 'official train',
    'validation_split': 'official val',
    'training_images_used': len(train_records),
    'validation_images_used': len(val_records),
    'metric': 'TorchMetrics MeanAveragePrecision at IoU=0.50',
    'faster_rcnn': {
        'iterations': FRCNN_MAX_ITER,
        **frcnn_metrics,
    },
    'detr': {
        'epochs': DETR_EPOCHS,
        **detr_metrics,
    },
}

comparison_path = OUTPUT_DIR / f'comparison_metrics_{RUN_NAME}.json'
comparison_path.write_text(json.dumps(comparison, indent=2))

print(json.dumps(comparison, indent=2))
print('Saved:', comparison_path)

## 11. Interpretation checklist

Before publishing the final metrics:

1. Confirm `QUICK_RUN` is `False`.
2. Confirm the full official validation split was evaluated for both models.
3. Keep the generated metrics JSON with the notebook output.
4. Include representative successes and failures, not only the best examples.
5. Discuss the training-budget difference: DETR generally converges more slowly than Faster R-CNN.

The original homework's DETR `AP50 = 56.3%` is intentionally not copied into this notebook because the supplied DETR notebook contains no matching evaluation output. Use only the metric generated above.